In [65]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence, PatternedSequenceGenerator, compute_bpc

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 
import os
import zipfile

In [66]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [67]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [68]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65
      
            self.y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [69]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, total_lags=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')

        self.linear = nn.ModuleDict({
            str(ii): nn.Linear(hidden_size, vocab_size) 
            for ii in range(total_lags)
        })

        ### use orthogonal initialization of weights ### 
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)                      # [batch, seq_len, emb_dim]
        out, h = self.rnn(embedded, h)                    # out: [batch, seq_len, hidden_size]
        
        last_hidden = out[:, -1, :]       # [batch, memory_unit]
        
        output_heads = {ii: head(last_hidden) 
                        for ii, head in self.linear.items()}
        
        return output_heads, h

In [70]:
def multihead_loss(output_heads, targets, criterion=None, reduction="mean"):
    """
    output_heads: dict[str, Tensor], each [batch, vocab_size]
    targets: list[Tensor] or dict[str, Tensor], each [batch]
    criterion: loss function, defaults to CrossEntropyLoss
    reduction: 'mean' or 'sum'
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    losses = []
    for key, pred in output_heads.items():
        # if targets is a list, assume order matches keys
        if isinstance(targets, dict):
            target = targets[key]
        else:
            target = targets[int(key)]   # cast string key to int if using list
        # print(pred, target)
        losses.append(criterion(pred[0], target))

    if reduction == "mean":
        return sum(losses) / len(losses)
    elif reduction == "sum":
        return sum(losses)
    else:
        return losses   # return list if you want per-head loss


In [74]:
short_term_memory = 20
total_samples = 1000000
working_memory = 1
hidden_size = 200
vocab_size = 27
embedding_dim = 30
lr = 5e-5


model = RNNEncoder(vocab_size, embedding_dim, hidden_size, total_lags=short_term_memory)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
criterion = torch.nn.CrossEntropyLoss()

data = get_random_sequence(total_samples, token_number=vocab_size)#get_sequence(total_samples, 2, 3, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

total = 0
test_acc = []
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        y_pred, h = model(X)
    else:
        y_pred, h = model(X, hidden)

    loss = multihead_loss(y_pred, X[0])     
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        hidden = h.detach()
        total += 1

        key = str(short_term_memory-1)
        if X[0][-1] == y_pred[key].argmax():
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0
        
        if total%10000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {np.sum(correct)/total if total<1000 else np.sum(correct)/1000:.4f}')

    

Iter : 10001, loss: 2.8756, accuracy: 0.9980
Iter : 20001, loss: 2.6940, accuracy: 1.0000
Iter : 30001, loss: 2.4719, accuracy: 1.0000
Iter : 40001, loss: 2.2132, accuracy: 1.0000
Iter : 50001, loss: 2.1641, accuracy: 1.0000
Iter : 60001, loss: 2.1990, accuracy: 1.0000
Iter : 70001, loss: 1.9070, accuracy: 1.0000
Iter : 80001, loss: 2.0259, accuracy: 1.0000
Iter : 90001, loss: 1.7882, accuracy: 1.0000
Iter : 100001, loss: 1.5827, accuracy: 1.0000
Iter : 110001, loss: 1.6702, accuracy: 1.0000
Iter : 120001, loss: 1.8130, accuracy: 1.0000
Iter : 130001, loss: 1.6031, accuracy: 1.0000
Iter : 140001, loss: 1.7334, accuracy: 1.0000
Iter : 150001, loss: 1.4838, accuracy: 1.0000
Iter : 160001, loss: 1.3227, accuracy: 1.0000
Iter : 170001, loss: 1.5483, accuracy: 1.0000
Iter : 180001, loss: 1.3886, accuracy: 1.0000
Iter : 190001, loss: 1.3634, accuracy: 1.0000
Iter : 200001, loss: 1.2386, accuracy: 1.0000
Iter : 210001, loss: 1.3756, accuracy: 1.0000
Iter : 220001, loss: 1.2005, accuracy: 1.00

In [75]:
with open('pretrained_model.pickle', 'wb') as f:
    pickle.dump(model, f)

In [78]:
# Step 1: Download and extract text8
def download_text8(path="dataset/text8.zip"):
    url = "http://mattmahoney.net/dc/text8.zip"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not os.path.exists(path):
        print("Downloading text8...")
        urllib.request.urlretrieve(url, path)
    with zipfile.ZipFile(path) as zf:
        data = zf.read(zf.namelist()[0]).decode('utf-8')
    return data

# Step 2: Build character-level vocabulary
def build_vocab(text):
    chars = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

# Step 3: Encode text into integer tokens
def encode(text, stoi):
    return np.array([stoi[c] for c in text], dtype=np.int32)

# Step 4: Create input and target sequences
#def create_dataset(encoded_text, seq_len=64, step=1):
#    inputs, targets = [], []
#    for i in range(0, len(encoded_text) - seq_len, step):
#        inputs.append(encoded_text[i:i+seq_len])
#        targets.append(encoded_text[i+1:i+seq_len+1])
#    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

class Dataset_converter(Dataset):
    def __init__(self, encoded_text, working_memory=1, short_term_memory=8):
        
        self.X = []
        self.y = []

        for ii in range(0, len(encoded_text) - working_memory - short_term_memory, 1):
            self.X.append(
                encoded_text[ii:ii+short_term_memory]
            )
            self.y.append(
                encoded_text[ii+short_term_memory]
            )

        self.X = tnsr(np.array(self.X)).long()
        self.y = tnsr(np.array(self.y)).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [102]:
class RNN_mem2(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size_pattern, hidden_size_mem):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn_mem = nn.RNN(embedding_dim, hidden_size_mem, batch_first=True, nonlinearity='tanh')


        self.linear = nn.Linear(hidden_size_mem, hidden_size_pattern)
        self.output_linear = nn.Linear(hidden_size_pattern, vocab_size)

        with open('pretrained_model.pickle', 'rb') as f:
            pretrained_model = pickle.load(f)
        
        with torch.no_grad():
            for p2, p1 in zip(self.rnn_mem.parameters(), pretrained_model.rnn.parameters()):
                p2.copy_(p1)
        
        
            for p2, p1 in zip(self.embedding.parameters(), pretrained_model.embedding.parameters()):
                p2.copy_(p1)
        
        for param in self.rnn_mem.parameters():
            param.requires_grad = False

        for param in self.embedding.parameters():
            param.requires_grad = False

    
    def forward(self, x, h_mem=None):
        embedded = self.embedding(x)                      # [batch, seq_len, emb_dim]
        _, h_mem = self.rnn_mem(embedded, h_mem)                    # out: [batch, seq_len, hidden_size]
        h_mem = h_mem.detach()  # just in case
        
        out = torch.nn.functional.relu(self.linear(h_mem))
        output_head = self.output_linear(out)
        
        return output_head[:,-1,:], h_mem

In [38]:
text = download_text8()
stoi, itos = build_vocab(text)
encoded = encode(text, stoi)
train_sample = 90000000
test_sample = 10000000
short_term_memory = 1

train_data_set = Dataset_converter(encoded[:train_sample], 1, short_term_memory=short_term_memory)
test_data_set = Dataset_converter(encoded[train_sample:], 1, short_term_memory=short_term_memory)

In [103]:
reps = 1
lr = 1e-4
total_samples = 100000
working_memory = 1
hidden_size_mem = 200
hidden_size_pattern = 100
vocab_size = 27
embedding_dim = 30

for rep in range(reps):
    model = RNN_mem2(vocab_size, embedding_dim, hidden_size_pattern, hidden_size_mem)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    criterion = torch.nn.CrossEntropyLoss()
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10000)

    total = 0
    total_avg = 10000
    train_loader = DataLoader(train_data_set, batch_size=1, shuffle=False)
    bpc = 0
    acc = 0

    for X, y in train_loader:
        optimizer.zero_grad()

        if total == 0:
            y_pred, h_mem = model(X)
        else:
            y_pred, h_mem = model(X, hidden_mem)

        loss = criterion(y_pred, y)     
        loss.backward()
        optimizer.step()

        # print(total)
        with torch.no_grad():
            hidden_mem = h_mem.detach()

            total += 1

            if y[0] == y_pred[0].argmax():
                    acc += 1
            
            bpc += compute_bpc(y_pred[0], y)
            
            
            if total%total_avg == 0:
                print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {acc/total_avg:.4f}, bpc: {bpc/total_avg:.4f}')
                acc = 0
                bpc = 0
                lr_scheduler.step()

Iter : 10001, loss: 1.9085, accuracy: 0.1968, bpc: 4.0461
Iter : 20001, loss: 4.0205, accuracy: 0.2549, bpc: 3.6994
Iter : 30001, loss: 1.2484, accuracy: 0.2961, bpc: 3.5123
Iter : 40001, loss: 1.4615, accuracy: 0.3027, bpc: 3.4566
Iter : 50001, loss: 2.2598, accuracy: 0.2951, bpc: 3.4269
Iter : 60001, loss: 2.7853, accuracy: 0.2834, bpc: 3.4761
Iter : 70001, loss: 2.0736, accuracy: 0.3065, bpc: 3.3902
Iter : 80001, loss: 2.3709, accuracy: 0.3286, bpc: 3.2922
Iter : 90001, loss: 3.6105, accuracy: 0.3284, bpc: 3.3106


KeyboardInterrupt: 

In [18]:
y_pred

tensor([[[ 0.0302, -0.0559,  0.0323,  0.0578, -0.0745, -0.0332, -0.0751,
           0.0280,  0.0565, -0.0432, -0.0330,  0.0592, -0.0328,  0.0197,
          -0.0184,  0.0395,  0.0995, -0.0659, -0.0254,  0.0317, -0.0522,
           0.0010,  0.0855,  0.0542,  0.0484, -0.0429, -0.0268]]],
       grad_fn=<ViewBackward0>)

In [ ]:
class RNN_mem(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size_pattern, hidden_size_mem):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn_mem = nn.RNN(embedding_dim, hidden_size_mem, batch_first=True, nonlinearity='tanh')
        self.rnn_pattern = nn.RNN(embedding_dim, hidden_size_pattern, batch_first=True, nonlinearity='tanh')


        self.w_fp = nn.Linear(hidden_size_mem, embedding_dim)
        self.output_linear = nn.Linear(hidden_size_pattern, vocab_size)

        with open('pretrained_model.pickle', 'rb') as f:
            pretrained_model = pickle.load(f)
        
        with torch.no_grad():
            for p2, p1 in zip(self.rnn_mem.parameters(), pretrained_model.rnn.parameters()):
                p2.copy_(p1)
        
        
            for p2, p1 in zip(self.embedding.parameters(), pretrained_model.embedding.parameters()):
                p2.copy_(p1)
        
        for param in self.rnn_mem.parameters():
            param.requires_grad = False

        for param in self.embedding.parameters():
            param.requires_grad = False
                
        ### use orthogonal initialization of weights ### 
        for name, param in self.rnn_pattern.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h_mem=None, h_pattern=None):
        embedded = self.embedding(x)                      # [batch, seq_len, emb_dim]
        out_mem, h_mem = self.rnn_mem(embedded, h_mem)                    # out: [batch, seq_len, hidden_size]
        h_mem = h_mem.detach()  # just in case
        
        frozen_to_plastic = self.w_fp(out_mem)
        out, h_pattern = self.rnn_pattern(frozen_to_plastic, h_pattern)#(embedded+frozen_to_plastic, h_pattern)

        last_hidden = out[:, -1, :]       # [batch, memory_unit]
        output_head = self.output_linear(last_hidden)
        
        return output_head, h_mem, h_pattern

In [209]:
text = download_text8()
stoi, itos = build_vocab(text)
encoded = encode(text, stoi)
train_sample = 90000000
test_sample = 10000000
short_term_memory = 1

train_data_set = Dataset_converter(encoded[:train_sample], 1, short_term_memory=short_term_memory)
test_data_set = Dataset_converter(encoded[train_sample:], 1, short_term_memory=short_term_memory)

In [90]:
reps = 1
lr = 1e-4
total_samples = 100000
working_memory = 1
hidden_size_mem = 200
hidden_size_pattern = 624
vocab_size = 27
embedding_dim = 30

for rep in range(reps):
    model = RNN_mem(vocab_size, embedding_dim, hidden_size_pattern, hidden_size_mem)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
    criterion = torch.nn.CrossEntropyLoss()
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10000)

    total = 0
    total_avg = 10000
    train_loader = DataLoader(train_data_set, batch_size=1, shuffle=False)
    bpc = 0
    acc = 0

    for X, y in train_loader:
        optimizer.zero_grad()

        if total == 0:
            y_pred, h_mem, h_pattern = model(X)
        else:
            y_pred, h_mem, h_pattern = model(X, hidden_mem, hidden_pattern)

        loss = criterion(y_pred, y)     
        loss.backward()
        optimizer.step()

        # print(total)
        with torch.no_grad():
            hidden_mem = h_mem.detach()
            hidden_pattern = h_pattern.detach()

            total += 1

            if y[0] == y_pred[0].argmax():
                    acc += 1
            
            bpc += compute_bpc(y_pred[0], y)
            
            
            if total%total_avg == 0:
                print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {acc/total_avg:.4f}, bpc: {bpc/total_avg:.4f}')
                acc = 0
                bpc = 0
                lr_scheduler.step()

Iter : 10001, loss: 1.1176, accuracy: 0.2885, bpc: 3.5372


KeyboardInterrupt: 

In [121]:
X

tensor([[3]])

In [ ]:
total = 0
bpc_test = 0
test_loader = DataLoader(test_data_set, batch_size=1, shuffle=False)
h_mem = None
h_pattern = None 

for X, y in tqdm(test_loader):
    with torch.no_grad():
        logits, h_mem, h_pattern = model(X, h_mem, h_pattern) 

        bpc_test += compute_bpc(logits[0], y)

        total += 1
        
print(f'Finall BPC on test set: {bpc_test/total:.4f}')

In [106]:
with open('pretrained_model.pickle', 'rb') as f:
    model = pickle.load(f)

In [113]:
#h = None

with torch.no_grad():
    for ii in range(7):
        # num = np.random.randint(0,8)
        # print(num)
        y_pred, h = model(torch.tensor([[ii]]), h)

In [114]:
for ii in range(20):
    print(ii, y_pred[str(ii)].argmax())

0 tensor(1)
1 tensor(2)
2 tensor(3)
3 tensor(4)
4 tensor(5)
5 tensor(6)
6 tensor(0)
7 tensor(1)
8 tensor(2)
9 tensor(3)
10 tensor(4)
11 tensor(5)
12 tensor(6)
13 tensor(0)
14 tensor(1)
15 tensor(2)
16 tensor(3)
17 tensor(4)
18 tensor(5)
19 tensor(6)


In [86]:
for name, param in model.named_parameters():
    print(name, param.requires_grad)

embedding.weight False
rnn_mem.weight_ih_l0 False
rnn_mem.weight_hh_l0 False
rnn_mem.bias_ih_l0 False
rnn_mem.bias_hh_l0 False
linear.weight True
linear.bias True
output_linear.weight True
output_linear.bias True


In [115]:
h

tensor([[[-0.4337,  0.1123, -0.2930,  0.4734, -0.6110,  0.0216, -0.2235,
           0.1747,  0.8774,  0.1822, -0.3149,  0.0845, -0.3353, -0.4770,
           0.2380,  0.0427,  0.3935,  0.9309,  0.2695,  0.0485, -0.2745,
           0.3782, -0.0985, -0.7911,  0.6333,  0.1008, -0.2400, -0.2645,
          -0.5950,  0.9004, -0.3214, -0.1446, -0.4683, -0.6055,  0.1743,
           0.0237, -0.1185,  0.0641, -0.4784,  0.3269,  0.8375, -0.0929,
          -0.0257,  0.6755,  0.3796, -0.3607,  0.3881,  0.0832,  0.0507,
           0.1377, -0.2377,  0.7350,  0.7771,  0.3376,  0.0091, -0.6539,
          -0.2545, -0.3696, -0.5937, -0.3933, -0.3620, -0.3772,  0.1640,
          -0.1821, -0.1772, -0.5215,  0.3739, -0.0491, -0.8844, -0.1900,
          -0.3057,  0.3096,  0.4002,  0.2863,  0.2198, -0.8493, -0.1419,
          -0.2007, -0.8094, -0.7855,  0.4673, -0.5797,  0.3426,  0.1186,
           0.4017, -0.6041,  0.0850, -0.1237,  0.0980,  0.3644, -0.6759,
           0.1199, -0.5179, -0.2209,  0.4503, -0.57

In [ ]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, use_embedding=True, embedding_dim=10):
        super().__init__()
        self.use_embedding = use_embedding

        if self.use_embedding:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        else:
            self.input_projection = nn.Linear(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        
        self.linear = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ### 
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h=None):
        if self.use_embedding:
            embedded = self.embedding(x)
            out, h = self.rnn(embedded, h)
        else:
            embedded = self.input_projection(x)
            out, h = self.rnn(embedded, h)

        out = self.linear(out[:,-1,:])

        return out, h  
    
class RNNDecoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, use_embedding=True, embedding_dim=10):
        super().__init__()
        self.use_embedding = use_embedding

        if self.use_embedding:
            self.embedding = nn.Embedding(vocab_size, embedding_dim)
            self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        else:
            self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True, nonlinearity='tanh')

        self.fc = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ###
        for name, param in self.rnn.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    
    def forward(self, x, h):
        if self.use_embedding:
            embedded = self.embedding(x)
            out, _ = self.rnn(embedded, h)
        else:
            out, _ = self.rnn(x, h)

        return self.fc(out) 
    
class RNNAutoencoder(nn.Module):
    def __init__(self, vocab_size, hidden_size, context_size, context_projection_size, use_embedding=True, embedding_dim=10 ):
        super().__init__()
        self.encoder = RNNEncoder(vocab_size, hidden_size, context_size, context_projection_size, use_embedding, embedding_dim)
        self.decoder = RNNDecoder(vocab_size, hidden_size, use_embedding, embedding_dim)
    
    def forward(self, x, decoder_input, context, h=None):
        next_token, h = self.encoder(x, context, h)
        decoder_output = self.decoder(decoder_input, h)
        return next_token, decoder_output, h

In [120]:
import torch.nn.functional as F

hs = []
h = None
with torch.no_grad():
    for ii in range(30):  # multiple cycles
        _, h = model(torch.tensor([[ii % 7]]), h)
        hs.append(h.squeeze(0).squeeze(0))

H = torch.stack(hs)
sim = F.cosine_similarity(H[:-7], H[7:], dim=-1)
print(sim.mean())  # should be high if 7-step periodic


tensor(0.7100)


In [118]:
H.size()

torch.Size([30, 200])